## Train data and ML Model Batch Process

In [2]:
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec


# Azure Open Dataset connection details
blob_account_name = "azureopendatastorage"
blob_container_name = "mlsamples"
blob_relative_path = "diabetes"
blob_sas_token = r""  # Blank  since container is Anonymous access

# Set Spark config to access blob storage
wasbs_path = f"wasbs://%s@%s.blob.core.windows.net/%s" % (blob_container_name, blob_account_name, blob_relative_path)
# spark.conf.set("fs.azure.sas.%s.%s.blob.core.windows.net" % (blob_container_name, blob_account_name), blob_sas_token)
spark.conf.set(f"fs.azure.sas.{blob_container_name}.{blob_account_name}.blob.core.windows.net", blob_sas_token)
print("Remote blob path :" + wasbs_path)

# Spark read parquet , note that it won't load any data yet by now
df = spark.read.parquet(wasbs_path).toPandas()

# Split the features and label from training

X,y = df[['AGE','SEX','BMI','BP','S1','S2','S3','S4','S5','S6']].values, df['Y'].values

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.30, random_state=0)

# Train the model in an MLFlow experiment 

experiment_name = "experiment-diabetes"
mlflow.set_experiment(experiment_name)
with mlflow.start_run():
    mlflow.autolog(log_models=False)

    model = DecisionTreeRegressor(max_depth=5)
    model.fit(X_train, y_train)

    # Define the model signature
    input_schema = Schema([

        ColSpec("integer", "AGE"),
        ColSpec("integer", "SEX"),
        ColSpec("double", "BMI"),
        ColSpec("double", "BP"),
        ColSpec("integer", "S1"),
        ColSpec("double", "S2"),
        ColSpec("double", "S3"),
        ColSpec("double", "S4"),
        ColSpec("double", "S5"),
        ColSpec("integer", "S6"),
    ])

    output_schema = Schema([ColSpec("integer")])
    signature = ModelSignature(inputs=input_schema, outputs=output_schema)

    # Log the model
    mlflow.sklearn.log_model(model, "model", signature=signature)

StatementMeta(, 2af678e7-61cc-47a7-9ee7-fab13d360e12, 4, Finished, Available, Finished, False)

Remote blob path :wasbs://mlsamples@azureopendatastorage.blob.core.windows.net/diabetes


2026/04/15 05:32:09 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


In [3]:
exp = mlflow.get_experiment_by_name(experiment_name)
last_run = mlflow.search_runs(exp.experiment_id, order_by=["start_time DESC"], max_results=1)
last_run_id = last_run.iloc[0]["run_id"]

print("Registering the model from run: ", last_run_id)
model_uri = "runs:/{}/model".format(last_run_id)
mv = mlflow.register_model(model_uri, "model-diabetes")
print("Name: {}".format(mv.name))
print("Version: {}".format(mv.version))

StatementMeta(, 2af678e7-61cc-47a7-9ee7-fab13d360e12, 5, Finished, Available, Finished, False)

Registering the model from run:  fad963d1-d1ad-403d-b2c4-ac68d9b74b28


2026-04-15:05:53:50,913 ERROR    [shared_platform_utils.py:82] Create MLModel failed, status_code: 400, b'{"requestId":"6e8127c3-4a47-4b97-963e-7e967a3bc11d","errorCode":"ItemDisplayNameAlreadyInUse","message":"Requested \'model-diabetes\' is already in use"}'
Registered model 'model-diabetes' already exists. Creating a new version of this model...
Created version '2' of model 'model-diabetes'.


In [ ]:
import mlflow
from synapse.ml.predict import MLFlowTransformer

df_test = spark.read.format("delta").load(f"Tables/{table_name}")

model = MLFlowTransformer(
    inputCols = ["AGE","SEX","BMI","BP","S1","S2","S3","S4","S5","S6"],
    outputCol = "predictions",
    modelName = "model-diabetes",
    modelVersion=1)

df_test = model.transform(df)

df_test.write.format('delta').mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)

In [ ]:
df = spark.sql("SELECT * FROM DiabetesData.diabetes_test LIMIT 1000")

display(df)